# Data (Pre)Processing

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn torch torchvision

In [43]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 

Wczytanie i przygotowanie danych

In [ ]:
data = pd.read_csv("./train_data.csv", delimiter=',')
data.head()

In [ ]:
data["SalePrice"] = data["SalePrice"].apply(lambda x: 0 if x <= 100000 else 1 if x <=350000 else 2)
data

Standaryzacja i tensory

In [56]:
data = pd.get_dummies(data, dtype=float)

X_df = data.drop(columns=["SalePrice"]) 
y = data["SalePrice"].values


X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)


# standaryzacja
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

Dataloader

In [57]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

Sieć - definicja

In [58]:
class Siec(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs):
        super(Siec, self).__init__()
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        return x

Sieć 

In [65]:
cols_num = X_train_tensor.shape[1]

model = Siec(num_inputs=cols_num, num_hidden=16, num_outputs=3).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
loss_module = nn.CrossEntropyLoss()


model.train() 

for epoch in range(500):
    epoch_loss = 0.0
    for data_inputs, data_labels in train_loader:
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)

        preds = model(data_inputs) 
        loss = loss_module(preds, data_labels)

        optimizer.zero_grad() 
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {epoch_loss/len(train_loader):.4f}")



Epoch: 000, Loss: 1.1319
Epoch: 010, Loss: 0.8186
Epoch: 020, Loss: 0.7145
Epoch: 030, Loss: 0.6478
Epoch: 040, Loss: 0.5909
Epoch: 050, Loss: 0.5471
Epoch: 060, Loss: 0.5125
Epoch: 070, Loss: 0.4845
Epoch: 080, Loss: 0.4715
Epoch: 090, Loss: 0.4506
Epoch: 100, Loss: 0.4389
Epoch: 110, Loss: 0.4270
Epoch: 120, Loss: 0.4180
Epoch: 130, Loss: 0.4111
Epoch: 140, Loss: 0.4062
Epoch: 150, Loss: 0.4033
Epoch: 160, Loss: 0.3917
Epoch: 170, Loss: 0.3874
Epoch: 180, Loss: 0.3823
Epoch: 190, Loss: 0.3783
Epoch: 200, Loss: 0.3782
Epoch: 210, Loss: 0.3705
Epoch: 220, Loss: 0.3693
Epoch: 230, Loss: 0.3642
Epoch: 240, Loss: 0.3619
Epoch: 250, Loss: 0.3582
Epoch: 260, Loss: 0.3552
Epoch: 270, Loss: 0.3573
Epoch: 280, Loss: 0.3497
Epoch: 290, Loss: 0.3463
Epoch: 300, Loss: 0.3491
Epoch: 310, Loss: 0.3432
Epoch: 320, Loss: 0.3468
Epoch: 330, Loss: 0.3416
Epoch: 340, Loss: 0.3365
Epoch: 350, Loss: 0.3337
Epoch: 360, Loss: 0.3378
Epoch: 370, Loss: 0.3310
Epoch: 380, Loss: 0.3290
Epoch: 390, Loss: 0.3264


Ewaluacja na zbiorze testowym

In [66]:
model.eval()


with torch.no_grad():
    test_preds = model(X_test_tensor)
    predicted_classes = torch.argmax(test_preds, dim=1)
    

    accuracies = []


    for i in range(3):
        given_class_obj = (y_test_tensor == i)
        total_in_class = given_class_obj.sum().item()
        
        if total_in_class > 0:
            class_correct = ((predicted_classes == i) & given_class_obj).sum().item()
            acc = class_correct / total_in_class
            accuracies.append(acc)


            print(f"Skutecznosc dla klasy {i}: {acc*100:.1f}%")
        else:
            print(f"Klasa {i} nie wystapila w zbiorze testowym.")


Skutecznosc dla klasy 0: 50.4%
Skutecznosc dla klasy 1: 93.6%
Skutecznosc dla klasy 2: 76.9%
